In [10]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import pearsonr
from sklearn.metrics import roc_auc_score, log_loss, mean_squared_error

# 1. Load the cleaned dataset AND drop any remaining NaNs
df = pd.read_csv("DD_master_dataset.csv").dropna().copy()

def scale_positive(x):
    """
    Max-scaling to ensure features are comparably sized for L2 regularization
    while strictly preserving positive values for the hyperbolic discounting formula.
    """
    return x / (np.max(x) + 1e-8)

print(f"Dataset successfully loaded and fully cleaned. Total rows: {len(df)}")
print(f"Total unique subjects: {df['subject_id'].nunique()}")

Dataset successfully loaded and fully cleaned. Total rows: 28838
Total unique subjects: 99


In [11]:
def sigmoid(x):
    """Sigmoid choice rule, clipped to prevent exponential overflow."""
    x = np.clip(x, -50, 50)
    return 1 / (1 + np.exp(-x))

def baseline_nll(params, r_imm, r_del, delay, choices, lambda_reg=0.1):
    """Negative Log-Likelihood for the Choice-Only Baseline."""
    kappa, beta = params
    if kappa <= 0: return 1e10

    v_imm = r_imm
    v_del = r_del / (1 + kappa * delay)
    delta_v = v_imm - v_del

    pi = np.clip(sigmoid(beta * delta_v), 1e-10, 1 - 1e-10)
    nll_choice = -np.sum(choices * np.log(pi) + (1 - choices) * np.log(1 - pi))

    reg = (lambda_reg / 2) * (kappa**2 + beta**2)
    return nll_choice + reg

def joint_nll(params, r_imm, r_del, delay, choices, log_rt, lambda_reg=0.1):
    """Negative Log-Likelihood for the Magnitude + Conflict Joint Model."""
    kappa, beta, beta0, beta1, beta2, sigma2 = params
    if kappa <= 0 or sigma2 <= 0: return 1e10

    v_imm = r_imm
    v_del = r_del / (1 + kappa * delay)
    delta_v = v_imm - v_del

    pi = np.clip(sigmoid(beta * delta_v), 1e-10, 1 - 1e-10)
    nll_choice = -np.sum(choices * np.log(pi) + (1 - choices) * np.log(1 - pi))

    mu = beta0 - beta1 * np.abs(delta_v) + beta2 * (v_imm + v_del)
    nll_rt = np.sum(0.5 * np.log(2 * np.pi * sigma2) + ((log_rt - mu)**2) / (2 * sigma2))

    reg = (lambda_reg / 2) * (kappa**2 + beta**2 + beta1**2 + beta2**2)
    return nll_choice + nll_rt + reg

In [12]:
results_list = []
subjects = df['subject_id'].unique()

for subj in subjects:
    subj_data = df[df['subject_id'] == subj]

    for run in ['train', 'test']:
        run_data = subj_data[subj_data['run_type'] == run]

        if len(run_data) == 0:
            continue

        # Use positive scaling to handle the L2 regularization fairly
        r_imm_scaled = scale_positive(run_data['r_imm'].values)
        r_del_scaled = scale_positive(run_data['r_del'].values)
        delay_scaled = scale_positive(run_data['delay'].values)

        choices = run_data['choice'].values
        log_rt = run_data['log_rt'].values

        # 1. Fit Choice-Only Baseline
        init_base = [0.05, 1.0]
        res_base = minimize(baseline_nll, init_base,
                            args=(r_imm_scaled, r_del_scaled, delay_scaled, choices, 0.1),
                            method='L-BFGS-B',
                            bounds=[(1e-5, 10), (-10, 10)])
        k_base, b_base = res_base.x

        # 2. Fit Joint RT Model (Magnitude + Conflict)
        init_joint = [0.05, 1.0, np.mean(log_rt), 0.0, 0.0, np.var(log_rt)]
        res_joint = minimize(joint_nll, init_joint,
                             args=(r_imm_scaled, r_del_scaled, delay_scaled, choices, log_rt, 0.1),
                             method='L-BFGS-B',
                             bounds=[(1e-5, 10), (-10, 10), (None, None), (-10, 10), (-10, 10), (1e-3, None)])
        k_joint, b_joint, b0, b1, b2, s2 = res_joint.x

        # Store results
        results_list.append({
            'subject_id': subj,
            'run_type': run,
            'k_base': k_base, 'b_base': b_base,
            'k_joint': k_joint, 'b_joint': b_joint,
            'b0': b0, 'b1': b1, 'b2': b2, 'sigma2': s2
        })

df_params = pd.DataFrame(results_list)
print("Parameter inference complete.")

Parameter inference complete.


In [13]:
evaluation_results = []

for subj in subjects:
    subj_params = df_params[df_params['subject_id'] == subj]

    # Extract Run A (Train) parameters
    train_params = subj_params[subj_params['run_type'] == 'train'].iloc[0]

    # Extract Run B (Test) data
    test_data = df[(df['subject_id'] == subj) & (df['run_type'] == 'test')]
    if len(test_data) == 0: continue

    # Loop through the two conditions: 1.0 (Reward) and 2.0 (Loss)
    for condition_val, condition_name in zip([1.0, 2.0], ['Reward', 'Loss']):
        cond_data = test_data[test_data['condition'] == condition_val]

        if len(cond_data) == 0: continue

        r_imm_test = scale_positive(cond_data['r_imm'].values)
        r_del_test = scale_positive(cond_data['r_del'].values)
        delay_test = scale_positive(cond_data['delay'].values)
        choices_test = cond_data['choice'].values
        log_rt_test = cond_data['log_rt'].values

        # --- Predict using Baseline Model ---
        v_del_base = r_del_test / (1 + train_params['k_base'] * delay_test)
        pi_base = np.clip(sigmoid(train_params['b_base'] * (r_imm_test - v_del_base)), 1e-10, 1 - 1e-10)
        baseline_log_loss = log_loss(choices_test, pi_base)

        # --- Predict using Joint Model ---
        v_del_joint = r_del_test / (1 + train_params['k_joint'] * delay_test)
        delta_v_joint = r_imm_test - v_del_joint

        pi_joint = np.clip(sigmoid(train_params['b_joint'] * delta_v_joint), 1e-10, 1 - 1e-10)
        joint_choice_log_loss = log_loss(choices_test, pi_joint)

        mu_rt = train_params['b0'] - train_params['b1'] * np.abs(delta_v_joint) + train_params['b2'] * (r_imm_test + v_del_joint)
        joint_rt_mse = mean_squared_error(log_rt_test, mu_rt)

        evaluation_results.append({
            'subject_id': subj,
            'condition': condition_name,
            'baseline_choice_log_loss': baseline_log_loss,
            'joint_choice_log_loss': joint_choice_log_loss,
            'joint_rt_mse': joint_rt_mse
        })

df_eval = pd.DataFrame(evaluation_results)

# Print metrics separated by condition
print("--- OUT-OF-SAMPLE PREDICTION (RUN B) ---")
for condition_name in ['Reward', 'Loss']:
    cond_eval = df_eval[df_eval['condition'] == condition_name]
    print(f"\nCONDITION: {condition_name}")
    print(f"Mean Baseline Choice Log-Loss: {cond_eval['baseline_choice_log_loss'].mean():.4f}")
    print(f"Mean Joint Choice Log-Loss:    {cond_eval['joint_choice_log_loss'].mean():.4f}")
    print(f"Mean Joint RT MSE:             {cond_eval['joint_rt_mse'].mean():.4f}")

--- OUT-OF-SAMPLE PREDICTION (RUN B) ---

CONDITION: Reward
Mean Baseline Choice Log-Loss: 0.6722
Mean Joint Choice Log-Loss:    0.6731
Mean Joint RT MSE:             0.2205

CONDITION: Loss
Mean Baseline Choice Log-Loss: nan
Mean Joint Choice Log-Loss:    nan
Mean Joint RT MSE:             nan


In [16]:
import numpy as np

def simulate_synthetic_data(r_imm, r_del, delay, true_params):
    """Generates synthetic choices and log-RTs using known ground-truth parameters."""
    k_true, b_true, b0_true, b1_true, b2_true, s2_true = true_params

    v_imm = r_imm
    v_del = r_del / (1 + k_true * delay)
    delta_v = v_imm - v_del

    pi = np.clip(sigmoid(b_true * delta_v), 1e-10, 1 - 1e-10)
    mu = b0_true - b1_true * np.abs(delta_v) + b2_true * (v_imm + v_del)

    synthetic_choices = np.random.binomial(n=1, p=pi)
    synthetic_log_rt = np.random.normal(loc=mu, scale=np.sqrt(s2_true))

    return synthetic_choices, synthetic_log_rt

print("--- PARAMETER RECOVERY SIMULATION ---")

# Extract the real experimental design from a sample subject
sample_subj = df[df['subject_id'] == 'Subj_01']
r_imm_sim = scale_positive(sample_subj['r_imm'].values)
r_del_sim = scale_positive(sample_subj['r_del'].values)
delay_sim = scale_positive(sample_subj['delay'].values)

# Define a REALISTIC RANGE of True Parameters: [kappa, beta, beta0, beta1, beta2, sigma2]
# Profile 1: Highly Impulsive (high kappa), sharp choices (high beta)
# Profile 2: Patient (low kappa), noisy choices (low beta)
true_params_list = [
    {'name': 'Highly Impulsive', 'params': [0.8, 6.0, 7.2, 1.0, 0.0, 0.2]},
    {'name': 'Patient / Noisy',  'params': [0.1, 3.0, 7.5, 0.5, 0.2, 0.1]}
]

num_simulations = 20
np.random.seed(42) # Ensure reproducible results for your report

for profile in true_params_list:
    print(f"\nEvaluating Profile: {profile['name']}")
    tp = profile['params']
    print(f"TRUE PARAMETERS: kappa={tp[0]}, beta={tp[1]}")

    k_joint_rec, b_joint_rec = [], []
    k_base_rec, b_base_rec = [], []

    for i in range(num_simulations):
        syn_choices, syn_log_rt = simulate_synthetic_data(r_imm_sim, r_del_sim, delay_sim, tp)

        # Recover via Joint Model (Magnitude + Conflict)
        res_j = minimize(joint_nll, [0.05, 1.0, 7.0, 0.0, 0.0, 0.1],
                         args=(r_imm_sim, r_del_sim, delay_sim, syn_choices, syn_log_rt, 0.1),
                         method='L-BFGS-B', bounds=[(1e-5, 10), (-10, 10), (None, None), (-10, 10), (-10, 10), (1e-3, None)])
        k_joint_rec.append(res_j.x[0])
        b_joint_rec.append(res_j.x[1])

        # Recover via Choice-Only Baseline
        res_b = minimize(baseline_nll, [0.05, 1.0],
                         args=(r_imm_sim, r_del_sim, delay_sim, syn_choices, 0.1),
                         method='L-BFGS-B', bounds=[(1e-5, 10), (-10, 10)])
        k_base_rec.append(res_b.x[0])
        b_base_rec.append(res_b.x[1])

    print(f"JOINT RECOVERY:    Mean kappa: {np.mean(k_joint_rec):.3f} (Std Dev: {np.std(k_joint_rec):.3f}) | Mean beta: {np.mean(b_joint_rec):.3f} (Std Dev: {np.std(b_joint_rec):.3f})")
    print(f"BASELINE RECOVERY: Mean kappa: {np.mean(k_base_rec):.3f} (Std Dev: {np.std(k_base_rec):.3f}) | Mean beta: {np.mean(b_base_rec):.3f} (Std Dev: {np.std(b_base_rec):.3f})")

--- PARAMETER RECOVERY SIMULATION ---

Evaluating Profile: Highly Impulsive
TRUE PARAMETERS: kappa=0.8, beta=6.0
JOINT RECOVERY:    Mean kappa: 0.967 (Std Dev: 0.219) | Mean beta: 5.228 (Std Dev: 0.836)
BASELINE RECOVERY: Mean kappa: 1.055 (Std Dev: 0.278) | Mean beta: 5.100 (Std Dev: 0.791)

Evaluating Profile: Patient / Noisy
TRUE PARAMETERS: kappa=0.1, beta=3.0
JOINT RECOVERY:    Mean kappa: 0.170 (Std Dev: 0.132) | Mean beta: 2.703 (Std Dev: 1.086)
BASELINE RECOVERY: Mean kappa: 0.246 (Std Dev: 0.322) | Mean beta: 2.807 (Std Dev: 0.969)


In [15]:
from scipy.stats import pearsonr

# 1. Separate the training and testing parameters
train_params = df_params[df_params['run_type'] == 'train'].set_index('subject_id')
test_params = df_params[df_params['run_type'] == 'test'].set_index('subject_id')

# 2. Find the subjects that have both Run A and Run B successfully fitted
common_subjects = train_params.index.intersection(test_params.index)
train_matched = train_params.loc[common_subjects]
test_matched = test_params.loc[common_subjects]

# 3. Calculate Test-Retest Reliability (Pearson correlation)
print("--- TEST-RETEST RELIABILITY (Run A vs Run B) ---")

# Joint Model Reliability
r_k_joint, p_k_joint = pearsonr(train_matched['k_joint'], test_matched['k_joint'])
r_b_joint, p_b_joint = pearsonr(train_matched['b_joint'], test_matched['b_joint'])
print("\nJOINT MODEL:")
print(f"Kappa (\u03ba) Reliability: r = {r_k_joint:.3f} (p-value = {p_k_joint:.3e})")
print(f"Beta (\u03b2) Reliability:  r = {r_b_joint:.3f} (p-value = {p_b_joint:.3e})")

# Baseline Model Reliability
r_k_base, p_k_base = pearsonr(train_matched['k_base'], test_matched['k_base'])
r_b_base, p_b_base = pearsonr(train_matched['b_base'], test_matched['b_base'])
print("\nBASELINE MODEL:")
print(f"Kappa (\u03ba) Reliability: r = {r_k_base:.3f} (p-value = {p_k_base:.3e})")
print(f"Beta (\u03b2) Reliability:  r = {r_b_base:.3f} (p-value = {p_b_base:.3e})")

--- TEST-RETEST RELIABILITY (Run A vs Run B) ---

JOINT MODEL:
Kappa (κ) Reliability: r = 0.172 (p-value = 8.809e-02)
Beta (β) Reliability:  r = 0.101 (p-value = 3.187e-01)

BASELINE MODEL:
Kappa (κ) Reliability: r = 0.180 (p-value = 7.408e-02)
Beta (β) Reliability:  r = 0.077 (p-value = 4.491e-01)
